# 02 · Joins, Subqueries and CTEs

In a real database, information is split across many tables: customers in one, invoices in another, products in a third. This avoids repeating the same customer address on every invoice. To answer most questions we have to **join** the tables back together, and joins are where the most expensive SQL mistakes happen.

**What you will learn here**

1. Keys: how tables point to each other
2. `INNER JOIN`, `LEFT JOIN` and the anti-join, worked out by hand
3. Joining many tables: revenue per artist
4. Self join: employees and their managers
5. The **fan-out trap**: a join that multiplied revenue by 9
6. Subqueries, `EXISTS` and the `NOT IN` + `NULL` trap
7. CTEs (`WITH`): writing long queries in readable steps

> **Colour guide:** $\color{red}{\text{red}}$ marks a warning, a trap or a wrong result. $\color{green}{\text{green}}$ marks a passed check or a correct result.

## The dataset: Chinook music store

| | |
|---|---|
| **Name** | Chinook sample database |
| **Source** | [lerocha/chinook-database](https://github.com/lerocha/chinook-database) on GitHub |
| **Licence** | MIT |
| **What it is** | A digital music store, modelled on a real one (iTunes-style): customers buy tracks, each purchase is an invoice with one line per track |

A copy is saved in `data/chinook.sqlite` (version from the `master` branch). The tables we use:

| Table | One row per | Key columns |
|---|---|---|
| `Customer` | customer (59) | `CustomerId`, `SupportRepId` → `Employee` |
| `Employee` | employee (8) | `EmployeeId`, `ReportsTo` → `Employee` |
| `Invoice` | purchase (412) | `InvoiceId`, `CustomerId` → `Customer`, `Total` |
| `InvoiceLine` | track bought in a purchase (2,240) | `InvoiceId` → `Invoice`, `TrackId` → `Track`, `UnitPrice`, `Quantity` |
| `Track` | song (3,503) | `TrackId`, `AlbumId` → `Album`, `GenreId` → `Genre` |
| `Album` | album (347) | `AlbumId`, `ArtistId` → `Artist` |
| `Artist` | artist (275) | `ArtistId`, `Name` |
| `Genre` | genre (25) | `GenreId`, `Name` |

> **Note:** this is a teaching database with generated sales (for example, every customer has 6 or 7 invoices). It is perfect for practising joins, but don't read business conclusions into it.

In [1]:
import sqlite3
import pandas as pd

# open the file read-only, so no query can change it by accident
con = sqlite3.connect("file:../data/chinook.sqlite?mode=ro", uri=True)

def run(query):
    return pd.read_sql_query(query, con)

run('''
SELECT name AS table_name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
''')

,table_name
0,Album
1,Artist
2,Customer
3,Employee
4,Genre
5,Invoice
6,InvoiceLine
7,MediaType
8,Playlist
9,PlaylistTrack


---
## 1. Keys

> **Theory.**
>
> - A **primary key** identifies each row uniquely: `Customer.CustomerId`. No two customers share one.
> - A **foreign key** is a column that points to the primary key of another table: `Invoice.CustomerId` says which customer made the purchase.
>
> A join matches rows where a foreign key equals a primary key.

The relationship type matters a lot for joins:

| Relationship | Example | Joining it… |
|---|---|---|
| **one-to-one** | an employee and their HR record | keeps the row count |
| **one-to-many** | one customer, many invoices | repeats the "one" side for each match |
| **many-to-many** | tracks and playlists (via `PlaylistTrack`) | can multiply rows a lot |

---
## 2. INNER, LEFT and anti-join, by hand

**Small example.** Two tiny tables.

| customers: id | name |
|---|---|
| 1 | Ana |
| 2 | Ben |
| 3 | Chloé |

| orders: order_id | customer_id | amount |
|---|---|---|
| 10 | 1 | 20 |
| 11 | 1 | 35 |
| 12 | 2 | 15 |
| 13 | 4 | 50 |

**Manual calculation.**

- **INNER JOIN** keeps only pairs that match on both sides: Ana–10, Ana–11, Ben–12. **3 rows.** Chloé (no order) and order 13 (customer 4 does not exist) disappear.
- **LEFT JOIN** (customers on the left) keeps **every** customer. Chloé appears once with NULL for the order columns. **4 rows.**
- **Anti-join** = LEFT JOIN, then keep rows where the right side is NULL: customers with **no** order. **1 row: Chloé.**

Notice that Ana appears **twice** after the join, once per order. That is the one-to-many effect.

In [2]:
small = sqlite3.connect(":memory:")
pd.DataFrame({"id": [1, 2, 3], "name": ["Ana", "Ben", "Chloé"]}).to_sql("customers", small, index=False)
pd.DataFrame({"order_id": [10, 11, 12, 13], "customer_id": [1, 1, 2, 4], "amount": [20, 35, 15, 50]}).to_sql("orders", small, index=False)

def run_small(query):
    return pd.read_sql_query(query, small)

print("INNER JOIN")
display(run_small('''
SELECT c.name, o.order_id, o.amount
FROM customers AS c
INNER JOIN orders AS o ON o.customer_id = c.id
'''))

print("LEFT JOIN")
display(run_small('''
SELECT c.name, o.order_id, o.amount
FROM customers AS c
LEFT JOIN orders AS o ON o.customer_id = c.id
'''))

print("Anti-join: customers with no order")
display(run_small('''
SELECT c.name
FROM customers AS c
LEFT JOIN orders AS o ON o.customer_id = c.id
WHERE o.order_id IS NULL
'''))

INNER JOIN


,name,order_id,amount
0,Ana,10,20
1,Ana,11,35
2,Ben,12,15


LEFT JOIN


,name,order_id,amount
0,Ana,10.0,20.0
1,Ana,11.0,35.0
2,Ben,12.0,15.0
3,Chloé,NaN,NaN


Anti-join: customers with no order


,name
0,Chloé


> **Check:** 3, 4 and 1 rows, as calculated by hand.

> **Note:** `c` and `o` are **table aliases**: short names so we can write `c.name` instead of `customers.name`. When two tables have a column with the same name, the alias says which one we mean.

There are two more join types: `RIGHT JOIN` (a LEFT JOIN with the tables swapped) and `FULL OUTER JOIN` (keep unmatched rows from both sides). SQLite supports both since version 3.39:

In [3]:
run_small('''
SELECT c.name, o.order_id, o.amount
FROM customers AS c
FULL OUTER JOIN orders AS o ON o.customer_id = c.id
''')

,name,order_id,amount
0,Ana,10.0,20.0
1,Ana,11.0,35.0
2,Ben,12.0,15.0
3,Chloé,NaN,NaN
4,None,13.0,50.0


Order 13 now appears with no customer name. In a real database that would be a data quality problem: an order pointing to a customer that doesn't exist (an **orphan** row).

---
## 3. Joining many tables: revenue per artist

**Question:** which artists earned the store the most money?

**Plan first.** The money is in `InvoiceLine` (`UnitPrice × Quantity`), but the artist name is three tables away:

`InvoiceLine` → (`TrackId`) → `Track` → (`AlbumId`) → `Album` → (`ArtistId`) → `Artist`

Each step is many-to-one (many lines per track, many tracks per album, many albums per artist), so the joins don't create extra rows. Then group by artist and sum.

In [4]:
run('''
SELECT ar.Name                                   AS artist,
       COUNT(*)                                  AS tracks_sold,
       ROUND(SUM(il.UnitPrice * il.Quantity), 2) AS revenue
FROM InvoiceLine AS il
JOIN Track  AS t  ON t.TrackId   = il.TrackId
JOIN Album  AS al ON al.AlbumId  = t.AlbumId
JOIN Artist AS ar ON ar.ArtistId = al.ArtistId
GROUP BY ar.ArtistId, ar.Name
ORDER BY revenue DESC
LIMIT 5
''')

,artist,tracks_sold,revenue
0,Iron Maiden,140,138.60
1,U2,107,105.93
2,Metallica,91,90.09
3,Led Zeppelin,87,86.13
4,Lost,41,81.59


Iron Maiden is the top artist by revenue. (`JOIN` on its own means `INNER JOIN`.)

> **Note:** we group by `ArtistId` **and** `Name`. Two different artists could share a name, but never an id.

### Anti-join on real data: tracks that never sold

In [5]:
run('''
SELECT COUNT(*)                                  AS tracks_never_sold,
       (SELECT COUNT(*) FROM Track)              AS all_tracks,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM Track), 1) AS pct_never_sold
FROM Track AS t
LEFT JOIN InvoiceLine AS il ON il.TrackId = t.TrackId
WHERE il.InvoiceLineId IS NULL
''')

,tracks_never_sold,all_tracks,pct_never_sold
0,1519,3503,43.4


1,519 of the 3,503 tracks (43%) were never bought. An INNER JOIN would have hidden them completely, which is why "what is **missing**?" questions always need a LEFT JOIN.

---
## 4. Self join: employees and their managers

> **Theory.** A table can be joined with **itself**. `Employee.ReportsTo` points to another row of `Employee`, so we use the table twice with two aliases: `e` for the employee, `m` for the manager.

In [6]:
run('''
SELECT e.FirstName || ' ' || e.LastName AS employee,
       e.Title,
       m.FirstName || ' ' || m.LastName AS manager
FROM Employee AS e
LEFT JOIN Employee AS m ON m.EmployeeId = e.ReportsTo
''')

,employee,Title,manager
0,Andrew Adams,General Manager,None
1,Nancy Edwards,Sales Manager,Andrew Adams
2,Jane Peacock,Sales Support Agent,Nancy Edwards
3,Margaret Park,Sales Support Agent,Nancy Edwards
4,Steve Johnson,Sales Support Agent,Nancy Edwards
5,Michael Mitchell,IT Manager,Andrew Adams
6,Robert King,IT Staff,Michael Mitchell
7,Laura Callahan,IT Staff,Michael Mitchell


`||` joins pieces of text together. We used a LEFT JOIN on purpose: the General Manager reports to nobody (`ReportsTo` is NULL), and an INNER JOIN would have dropped that row from the list.

---
## 5. The fan-out trap

**Question:** what is the store's total revenue?

The `Invoice` table has a `Total` column, so the answer is just `SUM(Total)`. But imagine we also need something from `InvoiceLine` in the same query, and we join first:

In [7]:
run('''
SELECT (SELECT ROUND(SUM(Total), 2) FROM Invoice) AS correct_revenue,
       (SELECT ROUND(SUM(i.Total), 2)
        FROM Invoice AS i
        JOIN InvoiceLine AS il ON il.InvoiceId = i.InvoiceId) AS revenue_after_join
''')

,correct_revenue,revenue_after_join
0,2328.6,20848.62


$\color{red}{\text{After the join the "revenue" is 20,848.62 instead of 2,328.60, about 9 times too big.}}$

**Why?** One invoice has many lines (one-to-many). The join repeats each invoice once per line, so its `Total` is added once per line too:

| InvoiceId | Total | lines | after the join, Total is counted |
|---|---|---|---|
| 1 | 1.98 | 2 | 2 times |
| 2 | 3.96 | 4 | 4 times |
| … | … | … | … |

No error message, no warning, a number that looks plausible. This is the most common way SQL gives wrong answers.

**How to avoid it:**

1. Before joining, know the relationship: one-to-one, one-to-many or many-to-many.
2. Sum at the **right level**: invoice totals from `Invoice`, line amounts from `InvoiceLine`.
3. If you need both, **aggregate first, then join** (with a CTE, section 7).
4. Check row counts before and after a join.

In [8]:
run('''
SELECT (SELECT COUNT(*) FROM Invoice)                    AS invoice_rows,
       (SELECT COUNT(*) FROM Invoice AS i
        JOIN InvoiceLine AS il ON il.InvoiceId = i.InvoiceId) AS rows_after_join
''')

,invoice_rows,rows_after_join
0,412,2240


412 rows became 2,240. A quick row count like this catches the problem before it reaches a report.

---
## 6. Subqueries

> **Theory.** A **subquery** is a query inside another query, in brackets. It can return:
>
> - **one value** (scalar subquery): `WHERE Total > (SELECT AVG(Total) FROM Invoice)`
> - **a list**, used with `IN` / `NOT IN`
> - **a table**, used in `FROM`
>
> A **correlated** subquery refers to the outer row and runs once per row.

**Question:** how many invoices are above the average invoice?

In [9]:
run('''
SELECT ROUND((SELECT AVG(Total) FROM Invoice), 2) AS average_invoice,
       COUNT(*)                                   AS invoices_above_average
FROM Invoice
WHERE Total > (SELECT AVG(Total) FROM Invoice)
''')

,average_invoice,invoices_above_average
0,5.65,179


### EXISTS

`EXISTS (subquery)` is true if the subquery returns at least one row. **Question:** which employees manage somebody?

In [10]:
run('''
SELECT m.FirstName || ' ' || m.LastName AS manager, m.Title
FROM Employee AS m
WHERE EXISTS (SELECT 1 FROM Employee AS e WHERE e.ReportsTo = m.EmployeeId)
''')

,manager,Title
0,Andrew Adams,General Manager
1,Nancy Edwards,Sales Manager
2,Michael Mitchell,IT Manager


### Trap: NOT IN with a NULL in the list

Now the opposite question: which employees manage **nobody**? The natural way to write it:

In [11]:
print("NOT IN:")
display(run('''
SELECT FirstName || ' ' || LastName AS employee
FROM Employee
WHERE EmployeeId NOT IN (SELECT ReportsTo FROM Employee)
'''))

print("NOT EXISTS:")
display(run('''
SELECT m.FirstName || ' ' || m.LastName AS employee
FROM Employee AS m
WHERE NOT EXISTS (SELECT 1 FROM Employee AS e WHERE e.ReportsTo = m.EmployeeId)
'''))

NOT IN:


,employee


NOT EXISTS:


,employee
0,Jane Peacock
1,Margaret Park
2,Steve Johnson
3,Robert King
4,Laura Callahan


$\color{red}{\text{NOT IN returns nothing at all}}$, while $\color{green}{\text{NOT EXISTS correctly finds the 5 employees who manage nobody}}$.

**Why?** The list `SELECT ReportsTo FROM Employee` contains a NULL (the General Manager). `x NOT IN (1, 2, NULL)` means `x <> 1 AND x <> 2 AND x <> NULL`, and `x <> NULL` is "unknown" (notebook 01). So the whole condition is never true, for any employee.

> **Key idea:** prefer `NOT EXISTS` to `NOT IN` with a subquery. Or filter the NULLs out: `NOT IN (SELECT ReportsTo FROM Employee WHERE ReportsTo IS NOT NULL)`.

---
## 7. CTEs: long queries in readable steps

> **Theory.** A **CTE** (common table expression) gives a name to a subquery with `WITH name AS (...)`. You can then use it like a table. Several CTEs can follow each other, each building on the previous one. It's like naming intermediate DataFrames in pandas instead of writing one giant expression.

**Question:** for each support employee, how many customers do they look after, and how much revenue do those customers bring?

**Plan:**

1. `customer_revenue`: revenue per customer (aggregate `Invoice` **first**, to avoid the fan-out)
2. Join it to `Customer` (one row per customer, so no fan-out) and to `Employee`
3. Group by employee

In [12]:
run('''
WITH customer_revenue AS (
    SELECT CustomerId, SUM(Total) AS revenue
    FROM Invoice
    GROUP BY CustomerId
)
SELECT e.FirstName || ' ' || e.LastName AS support_rep,
       COUNT(*)                         AS customers,
       ROUND(SUM(cr.revenue), 2)        AS revenue,
       ROUND(AVG(cr.revenue), 2)        AS revenue_per_customer
FROM Customer AS c
JOIN customer_revenue AS cr ON cr.CustomerId = c.CustomerId
JOIN Employee AS e          ON e.EmployeeId  = c.SupportRepId
GROUP BY e.EmployeeId
ORDER BY revenue DESC
''')

,support_rep,customers,revenue,revenue_per_customer
0,Jane Peacock,21,833.04,39.67
1,Margaret Park,20,775.40,38.77
2,Steve Johnson,18,720.16,40.01


**Check:** the three revenue numbers should add up to the total revenue from section 5 (2,328.60). If they did not, a join would have lost or duplicated rows.

In [13]:
check = run('''
WITH customer_revenue AS (
    SELECT CustomerId, SUM(Total) AS revenue FROM Invoice GROUP BY CustomerId
)
SELECT ROUND(SUM(cr.revenue), 2) AS total_over_reps
FROM Customer AS c
JOIN customer_revenue AS cr ON cr.CustomerId = c.CustomerId
JOIN Employee AS e          ON e.EmployeeId  = c.SupportRepId
''')
print("sum over support reps:", check["total_over_reps"][0], "| total revenue: 2328.6")

sum over support reps: 2328.6 | total revenue: 2328.6


$\color{green}{\text{The totals match}}$, so every customer was counted exactly once.

---
## Summary

| Tool | Use it for | Watch out for |
|---|---|---|
| `INNER JOIN` | rows that match on both sides | silently drops unmatched rows |
| `LEFT JOIN` | keep every row of the left table | NULLs in the right-side columns |
| Anti-join | "which X have no Y?" | filter on a right-side key `IS NULL` |
| Self join | rows pointing to rows of the same table | needs two aliases |
| One-to-many join | adding detail rows | **fan-out**: sums at the wrong level get multiplied |
| Subquery | a value or list computed by another query | `NOT IN` + NULL returns nothing |
| `EXISTS` / `NOT EXISTS` | "is there at least one…" | safest choice for "has no…" |
| CTE (`WITH`) | readable step-by-step queries | aggregate first, then join |

**Habit:** check row counts and totals before and after every join.

## What's next

In notebook 03 we learn **window functions**: rankings, running totals and month-over-month changes, without collapsing rows like `GROUP BY` does.

## References and further reading

- [SQLite documentation: SELECT and joins](https://www.sqlite.org/lang_select.html) · [SQLite: WITH clause](https://www.sqlite.org/lang_with.html)
- [PostgreSQL tutorial: joins between tables](https://www.postgresql.org/docs/current/tutorial-join.html)
- [Chinook database on GitHub](https://github.com/lerocha/chinook-database)